### Cell 1 - 初始化環境
---
本程式碼用於初始化遙測分析環境，並連接 Microsoft Planetary Computer 的 STAC API。  
透過 `robust_search()` 函式搜尋指定時空範圍內的 Sentinel-2 L2A 影像，並依雲量進行篩選與排序。  
同時支援重試機制以提升搜尋穩定性。  
研究區域（bbox）與使用波段可透過 `.env` 檔進行彈性設定。123

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pystac_client
import planetary_computer as pc
import stackstac
import rioxarray as rxr
import xarray as xr
import geopandas as gpd
from shapely.geometry import Point

# Microsoft JhengHei for Chinese labels
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# Connect to Planetary Computer
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

def robust_search(bbox, datetime_range, cloud_max=30, max_items=60, tries=3):
    for attempt in range(tries):
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox,
                datetime=datetime_range,
                max_items=max_items,
            )
            items = list(search.items())
            items = [i for i in items if i.properties.get("eo:cloud_cover", 100) < cloud_max]
            items.sort(key=lambda i: i.properties["eo:cloud_cover"])
            return items
        except Exception as e:
            time.sleep(2 ** attempt)
    raise RuntimeError("STAC search failed 3x")

# AOI — Matai'an catchment (upper Wanrong → downstream Guangfu)
# 從 .env 讀取範圍和波段
bbox_str = os.getenv('MATAIAN_BBOX', '121.28,23.56,121.52,23.76')
bands_str = os.getenv('S2_BANDS', 'B02,B03,B04,B08,B11,B12')
 
MATAIAN_BBOX = [float(x) for x in bbox_str.split(',')]
WANTED_BANDS = bands_str.split(',')

### Cell 2 - 事前影像選擇（Act 1）
---

在指定時間範圍（2025-06-01 至 2025-07-15）與雲量門檻（20%）下，僅找到一張符合條件的 Sentinel-2 影像。儘管候選影像數量有限，該影像在馬太鞍溪谷研究區域內雲量極少，河道與周邊植被清晰可見，因此適合作為後續變化分析的事前基準影像。

In [ ]:
# [S2] ACT 1 — Pre-event STAC search (Jun 2025, before Typhoon Wipha)
# 從 .env 讀取日期範圍
pre_datetime_range = f"{os.getenv('PRE_EVENT_START')}/{os.getenv('PRE_EVENT_END')}"
cloud_max_pre = int(os.getenv('S2_CLOUD_MAX', 20))

items_pre = robust_search(MATAIAN_BBOX, pre_datetime_range, cloud_max=cloud_max_pre)

# TCI Quick-QA the top 3 candidates in a dynamic 1xN subplot grid
n = max(min(len(items_pre), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for ax, item in zip(axes, items_pre[:n]):
    href = item.assets["visual"].href
    
    # 🔑 重要：要 sign（Planetary Computer）
    signed_href = pc.sign(href)
    
    img = rxr.open_rasterio(signed_href, overview_level=3)
    
    # 轉成 (H, W, 3)
    img = img.transpose("y", "x", "band")
    
    ax.imshow(img)
    ax.set_title(f"Cloud: {item.properties['eo:cloud_cover']:.1f}%")
    ax.axis("off")

# Pick your best scene and save the item ID
pre_item = items_pre[0]
PRE_ITEM_ID = pre_item.id

### Cell 3 - 事件中影像選擇（Act 2）
---
在候選影像中，選擇 2025-08-17 作為事件中影像。雖然整體雲量約為 17.6%，但馬太鞍溪谷上游區域仍具良好可視性，可觀察到河道局部變寬與混濁水體特徵，顯示堰塞湖的形成跡象。相較之下，其餘影像受雲層影響較大或關鍵區域不清晰，因此未被選用。

In [ ]:
# [S3] ACT 2 — Mid-event STAC search (Aug–Sep 2025, barrier lake present, before Sep 23 breach)
# 從 .env 讀取日期範圍
mid_datetime_range = f"{os.getenv('MID_EVENT_START')}/{os.getenv('MID_EVENT_END')}"
cloud_max_mid = int(os.getenv('S2_CLOUD_MAX', 20))

items_mid = robust_search(MATAIAN_BBOX, mid_datetime_range, cloud_max=cloud_max_mid)

# TCI Quick-QA — cropped to AOI so the valley is visible
minx, miny, maxx, maxy = MATAIAN_BBOX  # [121.28, 23.56, 121.52, 23.76]

n = max(min(len(items_mid), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for ax, item in zip(axes, items_mid[:n]):
    signed_href = pc.sign(item.assets["visual"].href)

    # overview_level=2 → 中等解析度，讀完再裁切到馬太鞍範圍
    img = rxr.open_rasterio(signed_href, overview_level=2)
    img_crop = img.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy, crs="EPSG:4326")
    rgb = img_crop.transpose("y", "x", "band").values

    ax.imshow(rgb)
    ax.set_title(f"[{items_mid.index(item)}] {item.id[:24]}\nCloud: {item.properties['eo:cloud_cover']:.1f}%",
                 fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

# 看完上面的圖，填入你選的 index（0、1 或 2）
CHOSEN_MID_IDX = 1   # ← 改這裡
mid_item    = items_mid[CHOSEN_MID_IDX]
MID_ITEM_ID = mid_item.id
print(f"選定: [{CHOSEN_MID_IDX}] {MID_ITEM_ID}")
print(f"日期: {mid_item.properties['datetime'][:10]}  雲量: {mid_item.properties['eo:cloud_cover']:.1f}%")
print(f"搜尋範圍: {mid_datetime_range}  (來自 .env)")

### Cell 4 - 事件後影像選擇（Act 3）
---
在候選影像中，選擇 2025-10-01 作為事件後影像。選擇這張影像是因為在上游山谷中可以清楚看到一塊連續的藍綠色水體，顯示堰塞湖的形成，且雲量較低、不影響判讀。相比其他影像，這張最能明顯呈現濁水範圍與湖體形狀。

In [ ]:
# [S4] ACT 3 — Post-event STAC search (after Sep 23 breach)
# 從 .env 讀取日期範圍
post_datetime_range = f"{os.getenv('POST_EVENT_START')}/{os.getenv('POST_EVENT_END')}"
cloud_max_post = int(os.getenv('S2_CLOUD_MAX', 20))

items_post = robust_search(MATAIAN_BBOX, post_datetime_range, cloud_max=cloud_max_post)

# TCI Quick-QA top 3 — look for evidence of the drained lake bed + fresh sediment around Guangfu
n = max(min(len(items_post), 3), 1)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for ax, item in zip(axes, items_post[:n]):
    href = item.assets["visual"].href
    signed_href = pc.sign(href)
    
    img = rxr.open_rasterio(signed_href, overview_level=3)
    img = img.transpose("y", "x", "band")
    
    ax.imshow(img)
    ax.set_title(f"{item.id[:20]}\nCloud: {item.properties['eo:cloud_cover']:.1f}%")
    ax.axis("off")

# 看完上面的圖，填入你選的 index（0、1 或 2）
CHOSEN_POST_IDX = 1   # ← 改這裡
post_item    = items_post[CHOSEN_POST_IDX]
POST_ITEM_ID = post_item.id
print(f"選定: [{CHOSEN_POST_IDX}] {POST_ITEM_ID}")
print(f"日期: {post_item.properties['datetime'][:10]}  雲量: {post_item.properties['eo:cloud_cover']:.1f}%")
print(f"搜尋範圍: {post_datetime_range}  (來自 .env)")

### Cell 5 - 影像資料載入與預處理
---
本步驟使用 `stream_cube()` 函數從 STAC 讀取 Sentinel-2 影像資料，並透過 Planetary Computer 重新簽名以避免存取權限過期。資料以 `stackstac` 轉換為 xarray 資料結構，並限制於馬太鞍溪谷研究範圍，以提升運算效率。影像統一轉換至 UTM 座標系（EPSG:32651）與 10 公尺解析度，並將原始數值轉換為反射率（/10000）。

接著，利用 `composite_stretched()` 將選定波段組合為 RGB 影像，並透過百分位拉伸（2–98%）提升影像對比，使地表特徵（如水體、植被與裸地）更易於辨識。

最終分別建立事件前（Pre）、事件中（Mid）與事件後（Post）的影像資料集，作為後續變化分析與災害判釋的基礎。

In [ ]:
def stream_cube(item, bands=WANTED_BANDS, bbox=MATAIAN_BBOX):
    # 每次都重新簽名（避免 token 過期）
    signed = pc.sign(item)
    
    # 用 stackstac 把 STAC 變成 xarray cube
    cube = stackstac.stack(
        [signed],
        assets=bands,
        epsg=32651,              # 台灣UTM座標
        resolution=10,           # 10m解析度
        bounds_latlon=bbox,      # 限定範圍（加速）
        chunksize=2048           # dask chunk（避免爆記憶體）
    )
    
    # 移除 time 維度（因為只有一張）
    cube = cube.squeeze("time")
    
    # 轉 reflectance（Sentinel-2 原始值要 /10000）
    cube = cube / 10000.0
    
    return cube

def composite_stretched(cube, r, g, b, lo=2, hi=98):
    # 組 RGB
    rgb = np.stack([
        cube.sel(band=r),
        cube.sel(band=g),
        cube.sel(band=b)
    ], axis=-1)
    
    out = np.zeros_like(rgb, dtype=np.float32)
    
    # 每個通道做 percentile stretch
    for k in range(3):
        p_lo, p_hi = np.nanpercentile(rgb[..., k], [lo, hi])
        
        out[..., k] = np.clip(
            (rgb[..., k] - p_lo) / (p_hi - p_lo),
            0, 1
        )
    
    return out.astype(np.float32)

cube_pre  = stream_cube(pre_item)
cube_mid  = stream_cube(mid_item)
cube_post = stream_cube(post_item)

### Cell 6 - Three-Act TCI 對照
---
本圖展示事件前（Pre）、事件中（Mid）與事件後（Post）的真彩影像（TCI），用於視覺化比較災害發展過程。可觀察到中期上游出現堰塞湖，而後期下游區域則出現明顯的沉積與河道變化。

此步驟用於確認所選影像能清楚呈現關鍵事件階段，作為後續分析的基礎。

In [ ]:
# Three-Act TCI Panel
rgb_pre  = composite_stretched(cube_pre,  "B04", "B03", "B02")
rgb_mid  = composite_stretched(cube_mid,  "B04", "B03", "B02")
rgb_post = composite_stretched(cube_post, "B04", "B03", "B02")

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Three-Act TCI Panel — 馬太鞍溪堰塞湖事件", fontsize=14, fontweight="bold")

for ax, rgb, title in zip(axes,
    [rgb_pre, rgb_mid, rgb_post],
    [f"ACT 1 Pre\n{PRE_ITEM_ID[:24]}",
     f"ACT 2 Mid\n{MID_ITEM_ID[:24]}",
     f"ACT 3 Post\n{POST_ITEM_ID[:24]}"]):
    ax.imshow(rgb)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("output/three_act_TCI.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存：output/three_act_TCI.png")

### Cell 7 - 變化指標定義（Change Metrics）
---
本研究定義四種光譜變化指標，用於辨識水體、植被與裸地變化，進而分析災害影響：

- **NIR Drop（近紅外下降）**  
  定義：`pre(B08) - post(B08)`  
  說明：近紅外反射值下降通常代表植被減少或水體增加，可用於偵測淹水或植被破壞。

- **SWIR Post（短波紅外）**  
  定義：`post(B12)`  
  說明：短波紅外對土壤濕度與裸地敏感，可用於辨識崩塌區或濕潤土壤。

- **BSI（Bare Soil Index，裸地指標）**  
  定義：`((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))`  
  說明：數值越高代表裸地或土壤比例越高，常用於偵測崩塌與沉積區。

- **BSI Change（裸地變化）**  
  定義：`BSI(post) - BSI(pre)`  
  說明：正值代表裸地增加，可能對應沖刷或土石堆積。

- **NDVI（Normalized Difference Vegetation Index，植被指數）**  
  定義：`(B08 - B04) / (B08 + B04)`  
  說明：數值越高代表植被越茂密。

- **NDVI Change（植被變化）**  
  定義：`NDVI(post) - NDVI(pre)`  
  說明：負值代表植被減少，可能與崩塌或淹水有關。

這些指標將用於後續災害偵測（堰塞湖、崩塌與土石流）之判斷依據。

In [ ]:
# Four reusable change metric functions
def nir_drop(pre, post):
    # TODO: return pre.sel(band="B08") - post.sel(band="B08")
    return pre.sel(band="B08") - post.sel(band="B08")

def swir_post(post):
    # TODO: return post.sel(band="B12")
    return post.sel(band="B12")

def bsi(cube):
    # TODO: BSI = ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))
    # Bare Soil Index（裸地指標）
    B11 = cube.sel(band="B11")
    B04 = cube.sel(band="B04")
    B08 = cube.sel(band="B08")
    B02 = cube.sel(band="B02")
    return ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))

def bsi_change(pre, post):
    # TODO: return bsi(post) - bsi(pre)
    # 裸地變化（增加 = 可能沖刷/沉積）
    return bsi(post) - bsi(pre)

def ndvi(cube):
    # TODO: (B08 - B04) / (B08 + B04)
    # 植被指數
    B08 = cube.sel(band="B08")
    B04 = cube.sel(band="B04")
    return (B08 - B04) / (B08 + B04)

def ndvi_change(pre, post):
    # TODO: return ndvi(post) - ndvi(pre)
    # 植被減少
    return ndvi(post) - ndvi(pre)

### Cell 8 - 堰塞湖偵測（Barrier Lake Detection）
---
- 本步驟利用 Pre 與 Mid 影像之光譜變化，偵測上游堰塞湖的形成。首先選取近紅外（NIR）、藍光（Blue）與綠光（Green）波段，建立水體判別條件。由於堰塞湖為混濁水體，其近紅外反射值低於植被，但高於清水，因此設定多組 NIR 閾值（經測試後範圍落在0.19 ~ 0.195）進行測試。

- 透過經度門檻（121.33°E）限制分析範圍於上游區域，以避免將下游淹水誤判為堰塞湖。對每組閾值計算水體面積，並與 NCDR 參考值（約 0.86 km²）比較，選擇最接近者作為最佳閾值。

- 最終生成堰塞湖遮罩（lake mask），並疊加於影像上進行視覺驗證，確認偵測結果與實際地形變化一致。

In [ ]:
import os
from pyproj import Transformer

os.makedirs("output", exist_ok=True)

# [S7] Barrier lake detection
nir_pre   = cube_pre.sel(band="B08")
nir_mid   = cube_mid.sel(band="B08")
blue_mid  = cube_mid.sel(band="B02")
green_mid = cube_mid.sel(band="B03")

# 把 121.33°E 轉換成 UTM x（公尺），確保座標系一致
tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
x_gate, _ = tf.transform(121.33, 23.69)

# 直接用 xarray 的 x 座標做空間篩選（自動廣播到整張影像，不用 np.tile）
west_mask = cube_mid.x < x_gate

# --- 閥值測試 ---
lake_results = {}
for thr in [0.19, 0.193,0.194, 0.195]:
    mask = (
        (nir_pre   > 0.25) &   # 原本是植被
        (nir_mid   < thr)  &   # 變成水體
        (blue_mid  > 0.03) &   # 水的藍光特徵
        (green_mid > nir_mid) & # 水的光譜形狀
        west_mask              # 限制在上游（121.33°E 以西）
    )
    area_km2 = float(mask.sum().compute()) * 100 / 1e6
    lake_results[thr] = area_km2
    print(f"thr={thr}: {area_km2:.3f} km²")

# 選最接近 NCDR 參考值 0.86 km² 的閥值
best_thr = min(lake_results, key=lambda k: abs(lake_results[k] - 0.86))
print(f"\nBest threshold: {best_thr}  ({lake_results[best_thr]:.3f} km²)")

# --- 建立最終 lake_mask 並 compute ---
lake_mask = (
    (nir_pre   > 0.25) &
    (nir_mid   < best_thr) &
    (blue_mid  > 0.03) &
    (green_mid > nir_mid) &
    west_mask
).compute()

# --- 繪圖 ---
rgb_mid = composite_stretched(cube_mid, "B04", "B03", "B02")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("ACT 1→2 Barrier lake detection — NCDR ref ≈0.86 km²")

ax[0].imshow(rgb_mid)
ax[0].set_title("Mid-event TCI")
ax[0].axis("off")

ax[1].imshow(rgb_mid)
ax[1].imshow(np.where(lake_mask.values, 1, np.nan), cmap="cool", alpha=0.5)
ax[1].set_title(f"Lake mask  thr={best_thr}  {lake_results[best_thr]:.3f} km²")
ax[1].axis("off")

plt.tight_layout()
plt.savefig("output/lake_mask.png", dpi=150)
plt.show()


### Cell 9 - Pre → Mid 變化指標Change_maps生成（堰塞湖生成期）
---
本步驟計算四種光譜變化指標，以分析事件前（Pre）至事件中（Mid）期間的地表變化。透過 NIR Drop、SWIR、BSI Change 與 NDVI Change 的組合，可辨識水體生成、植被變化與地表裸露情形。

在堰塞湖形成階段，預期上游區域將出現近紅外反射下降（NIR Drop 增加）、NDVI 降低（植被減少），並伴隨水體相關的光譜特徵。SWIR 與 BSI 則輔助辨識濕潤土壤與可能的地表變動。

上述四個指標以 2×2 圖組呈現，提供後續水體偵測（Barrier Lake Mask）與閾值設定的重要依據。

In [ ]:
# [B-Mid] 四個 change metric — Pre → Mid（堰塞湖生成期）
import matplotlib.pyplot as plt
import numpy as np
import os
os.makedirs("output", exist_ok=True)

# --- 計算四個指標 ---
nd_mid   = nir_drop(cube_pre, cube_mid).compute()
sw_mid   = swir_post(cube_mid).compute()
bsi_mid  = bsi_change(cube_pre, cube_mid).compute()
ndvi_mid = ndvi_change(cube_pre, cube_mid).compute()

# --- 2×2 Change Map ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Pre → Mid  Change Maps（堰塞湖生成期）", fontsize=14)

maps = [
    (nd_mid,   "RdBu",   -0.3, 0.3,  "NIR Drop\n(pre B08 − mid B08)"),
    (sw_mid,   "YlOrBr",  0,   0.4,  "SWIR Post Brightness\n(mid B12)"),
    (bsi_mid,  "YlOrBr", -0.2, 0.2,  "BSI Change\n(mid − pre)"),
    (ndvi_mid, "RdYlGn", -0.4, 0.4,  "NDVI Change\n(mid − pre)"),
]

for ax, (data, cmap, vmin, vmax, title) in zip(axes.flatten(), maps):
    im = ax.imshow(data.values, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("output/B_mid_change_maps.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存：output/B_mid_change_maps.png")

### Cell 10 - Ground Truth 建立與驗證（Landslide vs Stable）

本步驟建立崩塌（Y）與穩定植被（N）之地面真實點（各 10 點），來源包含影像判釋與空間位置選取。所有點位先以 WGS84（EPSG:4326）建立，並轉換至影像座標系（UTM）以確保與光譜資料一致。

接著，於各點位擷取 NDVI 變化與 BSI 變化值，用於驗證崩塌區與穩定區之光譜差異。理論上，崩塌區應呈現 NDVI 下降（植被減少）與 BSI 上升（裸地增加），而穩定區則應接近零變化。

此地面真實資料將用於後續閾值調整與混淆矩陣評估，以提升崩塌偵測結果的可靠性。

In [ ]:
import geopandas as gpd
import numpy as np


landslide_truth = [
    (121.3061635098438, 23.703992232549655, "Y"),
    (121.30731649919981, 23.70373198228068, "Y"),
    (121.30650331946988, 23.7032225700078, "Y"),
    (121.30742306972793, 23.702838894216715, "Y"),
    (121.30634639941334, 23.702475827189748, "Y"),
    (121.30753583100906, 23.702502138634376, "Y"),
    (121.30624920756094, 23.70205447975377, "Y"),
    (121.30732837110972, 23.702009254523936, "Y"),
    (121.30641365193624, 23.701563990112135, "Y"),
    (121.30648876143486, 23.701073039701075, "Y"),
]

stable_truth = [
    (121.3141492544933, 23.70105738662312, 'N'),
    (121.31499265345711, 23.70103790633488, 'N'),
    (121.31407994569585, 23.699803581003007, 'N'),
    (121.315352615308,   23.69998080517178, 'N'),
    (121.31383909560971, 23.698995429840817, 'N'),
    (121.31530763248944, 23.69898604025192, 'N'),
    (121.31403719438201, 23.698189553880898, 'N'),
    (121.31400041408163, 23.69730612879972, 'N'),
    (121.31516205836908, 23.696998732276644, 'N'),
    (121.31261645662217, 23.697070975314183, 'N')
]


# === 1️⃣ 建立 GeoDataFrame ===

gdf_landslide = gpd.GeoDataFrame(
    landslide_truth,
    columns=["lon", "lat", "label"],
    geometry=gpd.points_from_xy(
        [r[0] for r in landslide_truth],
        [r[1] for r in landslide_truth],
    ),
    crs="EPSG:4326",
)

gdf_stable = gpd.GeoDataFrame(
    stable_truth,
    columns=["lon", "lat", "label"],
    geometry=gpd.points_from_xy(
        [r[0] for r in stable_truth],
        [r[1] for r in stable_truth],
    ),
    crs="EPSG:4326",
)

# === 2️⃣ 轉座標系 ===
gdf_landslide = gdf_landslide.to_crs("EPSG:32651")
gdf_stable    = gdf_stable.to_crs("EPSG:32651")


# === 3️⃣ 定義 sampling function ===
def sample_points(gdf, raster, name="value"):
    values = []
    
    for geom in gdf.geometry:
        x, y = geom.x, geom.y
        
        try:
            val = raster.sel(x=x, y=y, method="nearest").values.item()
        except:
            val = np.nan
        
        values.append(val)
    
    gdf[name] = values
    return gdf


# === 4️選你要驗證的指標 ===
# 先 compute，避免迴圈內反覆觸發 dask
nd_c = nir_drop(cube_pre, cube_post).compute()
sw_c = swir_post(cube_post).compute()

gdf_landslide = sample_points(gdf_landslide, nd_c, "nir_drop")
gdf_landslide = sample_points(gdf_landslide, sw_c, "swir_post")
gdf_stable    = sample_points(gdf_stable,    nd_c, "nir_drop")
gdf_stable    = sample_points(gdf_stable,    sw_c, "swir_post")

print("=== Landslide ===")
print(gdf_landslide[["label","nir_drop","swir_post"]])
print("\n=== Stable ===")
print(gdf_stable[["label","nir_drop","swir_post"]])

### Cell 11 - Landslide Threshold Tuning（崩塌閾值優化）
---
本步驟針對崩塌偵測建立多組候選閾值組合（NIR Drop 與 SWIR），並利用地面真實點（10 個崩塌點與 10 個穩定點）進行評估。對每組閾值產生崩塌遮罩（landslide mask），並在所有樣本點進行抽樣，比較預測結果與實際標記。

透過計算混淆矩陣（TP、FP、TN、FN）與 F1 分數，評估各組閾值的分類表現，並選擇 F1 最高者作為最佳參數。結果以 1×5 視覺化呈現，清楚比較不同閾值組合的空間分布與偵測效果。

此過程將崩塌偵測由經驗閾值轉為資料驅動的最佳化結果，提升分類準確性與方法可信度。

In [ ]:
# [S8b] Threshold tuning — 6 candidate pairs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from matplotlib.patches import Rectangle
from pyproj import Transformer

# 崩塌源區只可能在上游（121.38°E 以西）
tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
x_upstream_gate, _ = tf.transform(121.38, 23.69)
upstream_gate = (cube_post.x < x_upstream_gate).compute()

candidate_thresholds = [
    (0.11, 0.21),
    (0.12, 0.22),
    (0.13, 0.23),
    (0.14, 0.24),
    (0.15, 0.25),   # baseline
]

# --- 預先 compute，避免迴圈內重複觸發 dask ---
nd        = nir_drop(cube_pre, cube_post).compute()
sw        = swir_post(cube_post).compute()
nir_pre_c = cube_pre.sel(band="B08").compute()
rgb_post  = composite_stretched(cube_post, "B04", "B03", "B02")

# --- 合併 truth points，建立 y_true ---
gdf_all = pd.concat([gdf_landslide, gdf_stable], ignore_index=True)
y_true = (gdf_all["label"] == "Y").astype(int).values

# --- 主迴圈 ---
results = []
masks = []

for nd_min, sw_min in candidate_thresholds:
    # 建立 landslide mask：記得加 upstream_gate
    lmask = (
        (nd > nd_min) &
        (sw > sw_min) &
        (nir_pre_c > 0.25) &
        upstream_gate
    ).compute()

    # 在 20 個 truth points 抽樣
    y_pred = []
    for geom in gdf_all.geometry:
        try:
            val = lmask.sel(x=geom.x, y=geom.y, method="nearest").values
            val = bool(np.asarray(val).squeeze())
        except Exception:
            val = False
        y_pred.append(int(val))

    y_pred = np.array(y_pred)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    f1 = f1_score(y_true, y_pred, zero_division=0)

    results.append({
        "nir_drop_min": nd_min,
        "swir_post_min": sw_min,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "F1": round(f1, 3),
    })
    masks.append(lmask)

# --- 輸出結果表 ---
results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
print(results_df.to_markdown(index=False))

best_orig_idx = max(range(len(results)), key=lambda i: results[i]["F1"])
best_row = results[best_orig_idx]

# --- 繪圖：6組就畫 2×3 ---
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
fig.suptitle("ACT 1→3 Landslide source mask — threshold grid", fontsize=13)

for i, (ax, (nd_min, sw_min), res, lmask) in enumerate(
    zip(axes, candidate_thresholds, results, masks)
):
    ax.imshow(rgb_post)
    ax.imshow(np.where(lmask.values, 1, np.nan), cmap="Reds", alpha=0.6)
    ax.set_title(
        f"nd>{nd_min}, sw>{sw_min}\n"
        f"F1={res['F1']:.2f}  TP={res['TP']} FP={res['FP']} FN={res['FN']}",
        fontsize=9,
    )
    ax.axis("off")

    # 最佳 F1 加紅框
    if i == best_orig_idx:
        rect = Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            fill=False, edgecolor="red", linewidth=4, clip_on=False
        )
        ax.add_patch(rect)

plt.tight_layout()
plt.savefig("output/08_landslide_threshold_grid.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"\n最佳組合: nir_drop > {best_row['nir_drop_min']}, "
    f"swir_post > {best_row['swir_post_min']}  "
    f"F1 = {best_row['F1']:.3f}"
)

### Cell 12 - Pre → Post 變化指標change map生成（崩塌+土石流）
---
本步驟分析事件前（Pre）至事件後（Post）之光譜變化，以辨識崩塌源區與下游土石流影響。透過 NIR Drop、SWIR、BSI Change 與 NDVI Change 四項指標，可有效反映植被破壞、裸地增加與濕潤土壤分布。

在崩塌區，預期出現顯著的 NDVI 下降（植被移除）與 BSI 上升（裸地暴露），同時伴隨較高的 SWIR 值，反映乾燥或新暴露的地表。NIR Drop 則進一步強化對植被消失區域的辨識能力。於下游區域，則可能觀察到較分散的 NDVI 下降與 BSI 增加，對應土石流沉積與河道擴張。

上述變化以 2×2 圖組呈現，作為後續崩塌偵測（C2）與土石流範圍判定（C3）的依據。

In [ ]:
# [B-Post] 四個 change metric — Pre → Post（崩塌＋土石流期）
import os
os.makedirs("output", exist_ok=True)

# --- 計算四個指標 ---
nd_post   = nir_drop(cube_pre, cube_post).compute()
sw_post   = swir_post(cube_post).compute()
bsi_post  = bsi_change(cube_pre, cube_post).compute()
ndvi_post = ndvi_change(cube_pre, cube_post).compute()

# --- 2×2 Change Map ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Pre → Post  Change Maps（崩塌＋土石流期）", fontsize=14)

maps = [
    (nd_post,   "RdBu",   -0.3, 0.3,  "NIR Drop\n(pre B08 − post B08)"),
    (sw_post,   "YlOrBr",  0,   0.4,  "SWIR Post Brightness\n(post B12)"),
    (bsi_post,  "YlOrBr", -0.2, 0.2,  "BSI Change\n(post − pre)"),
    (ndvi_post, "RdYlGn", -0.4, 0.4,  "NDVI Change\n(post − pre)"),
]

for ax, (data, cmap, vmin, vmax, title) in zip(axes.flatten(), maps):
    im = ax.imshow(data.values, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("output/B_post_change_maps.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 已儲存：output/B_post_change_maps.png")

### Cell 13 - 下游土石流偵測（Debris Flow Detection）以及three acts面積總結
---
本步驟針對馬太鞍溪出海口以東（121.33°E）之下游光復區域，偵測土石流與沉積影響。

#### 偵測規則

$$
\text{debris\_mask} = \underbrace{(\Delta\text{NDVI} > 0.25)}_{\text{植被大幅減少}} \cap \underbrace{(\Delta\text{BSI} > 0.10)}_{\text{裸土顯著增加}} \cap \underbrace{(\text{NIR}_\text{pre} > 0.20)}_{\text{事前為植被}} \cap \underbrace{(x > 121.35°E)}_{\text{下游空間限制}}
$$

#### 空間互斥處理

最終 debris mask 進一步排除已辨識之**堰塞湖**與**崩塌源區**像素，確保三類災害結果彼此獨立不重疊：

$$
\text{debris\_final} = \text{debris\_mask} \setminus (\text{lake\_mask} \cup \text{landslide\_mask})
$$

#### 結果說明

下游區域出現廣泛的 NDVI 下降與 BSI 上升，對應大面積泥沙沉積，形成「沉積扇（sediment sheet）」的空間分布。
影響面積遠大於上游崩塌源區，顯示潰決洪水攜帶大量土砂覆蓋光復鄉農田與聚落。

In [ ]:
from pyproj import Transformer

# --- 座標轉換：121.38°E → UTM x（公尺）---
tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
x_debris_gate, _ = tf.transform(121.38, 23.69)
downstream_gate = (cube_post.x > x_debris_gate).compute()

# --- 預先 compute 指標 ---
ndvi_diff  = (ndvi(cube_pre) - ndvi(cube_post)).compute()
bsi_diff   = bsi_change(cube_pre, cube_post).compute()
nir_pre_c  = cube_pre.sel(band="B08").compute()

# --- 從 S8b 取最佳 landslide_mask ---
landslide_mask = masks[best_orig_idx]
landslide_mask = (landslide_mask & ~lake_mask).compute()  # 確保排除堰塞湖
# --- Debris mask（獨立條件）---
debris_mask = (
    (ndvi_diff > 0.08) &
    (nir_pre_c > 0.18) &
    downstream_gate        # 鎖在 121.38°E 以東
)

# 排除已偵測的堰塞湖 & 崩塌源區
debris_mask = (debris_mask & ~lake_mask & ~landslide_mask).compute()

debris_area_km2 = float(debris_mask.sum()) * 100 / 1e6
print(f"Debris flow area: {debris_area_km2:.3f} km²")

# --- 繪圖 1×3 ---
rgb_post = composite_stretched(cube_post, "B04", "B03", "B02")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ACT 1→3 Downstream debris flow — Guangfu sediment sheet", fontsize=13)

im0 = axes[0].imshow(ndvi_diff.values, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
axes[0].set_title("NDVI change (pre − post)\n紅色 = 植被減少", fontsize=10)
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(bsi_diff.values, cmap="YlOrBr", vmin=-0.3, vmax=0.3)
axes[1].set_title("BSI change (post − pre)\n暖色 = 裸土增加", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(rgb_post)
axes[2].imshow(np.where(debris_mask.values, 1, np.nan), cmap="copper", alpha=0.6)
axes[2].set_title(
    f"Post TCI + debris mask\n"
    f"面積 = {debris_area_km2:.3f} km²  (下游 121.33°E 以東)",
    fontsize=10)
axes[2].axis("off")

plt.tight_layout()
plt.savefig("output/debris_mask.png", dpi=150, bbox_inches="tight")
plt.show()

# --- 三幕面積總結 ---
print(f"\n{'='*40}")
print(f"ACT 1→2  堰塞湖面積:    {lake_results[best_thr]:.3f} km²")
print(f"ACT 1→3  崩塌源區面積:  {float(landslide_mask.sum()) * 100 / 1e6:.3f} km²")
print(f"ACT 1→3  土石流面積:    {debris_area_km2:.3f} km²")
print(f"{'='*40}")

### Cell 14 - 三階段災害結果整合與向量化（Three-Act Evidence）
---
本步驟將先前建立的三種災害遮罩（堰塞湖、崩塌源區與土石流）進行向量化處理，轉換為多邊形資料，並依最小面積門檻（湖：3000 m²、崩塌：2000 m²、土石流：5000 m²）過濾雜訊，以保留具實際地理意義的區域。

向量化結果以 GeoPackage 格式儲存，分為三個圖層（barrier_lake、landslide_source、debris_flow），便於後續 GIS 分析與疊圖應用。

同時，透過 2×2 視覺化呈現三階段災害證據：  
- **ACT 1→2**：上游堰塞湖形成  
- **ACT 1→3**：崩塌源區辨識  
- **ACT 1→3**：下游土石流影響  
- 綜合疊圖檢視三者空間關係  

最後進行空間互斥檢查（overlap test），確認三類災害區域彼此不重疊，確保分類邏輯一致且結果具物理合理性。整體結果完整呈現由上游崩塌引發堰塞湖，進而影響下游沉積的連鎖災害過程。

In [ ]:
# [S10] Vectorize the three masks
from rasterio import features
from shapely.geometry import shape
landslide_mask = masks[best_orig_idx]
landslide_mask = (landslide_mask & ~lake_mask).compute()

def vectorize(mask_da, min_area):
    m = mask_da.astype("uint8").values
    polys = [shape(g) for g, v in
             features.shapes(m, mask=m.astype(bool), transform=mask_da.rio.transform())
             if v == 1]
    gdf = gpd.GeoDataFrame(geometry=polys, crs=mask_da.rio.crs)
    return gdf[gdf.area > min_area]

# --- Vectorize ---
lake_gdf       = vectorize(lake_mask,       min_area=3000)
landslides_gdf = vectorize(landslide_mask,  min_area=2000)
debris_gdf     = vectorize(debris_mask,     min_area=5000)

print(f"堰塞湖多邊形:  {len(lake_gdf)}")
print(f"崩塌源區多邊形: {len(landslides_gdf)}")
print(f"土石流多邊形:  {len(debris_gdf)}")

# --- 存成 GeoPackage ---
gpkg_path = "mataian_detections.gpkg"
lake_gdf.to_file(       gpkg_path, layer="barrier_lake",    driver="GPKG")
landslides_gdf.to_file( gpkg_path, layer="landslide_source", driver="GPKG", mode="a")
debris_gdf.to_file(     gpkg_path, layer="debris_flow",      driver="GPKG", mode="a")
print(f"\n已儲存 → {gpkg_path}")

# --- 底圖 ---
rgb_mid  = composite_stretched(cube_mid,  "B04", "B03", "B02")
rgb_post = composite_stretched(cube_post, "B04", "B03", "B02")

# --- 2×2 繪圖 ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Three-Act Evidence — 馬太鞍溪堰塞湖事件", fontsize=14, fontweight="bold")

# [0,0] Mid TCI + 堰塞湖（cyan）
axes[0, 0].imshow(rgb_mid)
axes[0, 0].imshow(np.where(lake_mask.values, 1, np.nan), cmap="cool", alpha=0.6)
axes[0, 0].set_title(
    f"ACT 1→2  Barrier lake\n"
    f"{len(lake_gdf)} polygons  ·  {float(lake_mask.sum()) * 100 / 1e6:.3f} km²",
    fontsize=10)
axes[0, 0].axis("off")

# [0,1] Post TCI + 崩塌源區（red）
axes[0, 1].imshow(rgb_post)
axes[0, 1].imshow(np.where(landslide_mask.values, 1, np.nan), cmap="Reds", alpha=0.6)
axes[0, 1].set_title(
    f"ACT 1→3  Landslide source\n"
    f"{len(landslides_gdf)} polygons  ·  {float(landslide_mask.sum()) * 100 / 1e6:.3f} km²",
    fontsize=10)
axes[0, 1].axis("off")

# [1,0] Post TCI + 土石流（copper）
axes[1, 0].imshow(rgb_post)
axes[1, 0].imshow(np.where(debris_mask.values, 1, np.nan), cmap="copper", alpha=0.6)
axes[1, 0].set_title(
    f"ACT 1→3  Debris flow\n"
    f"{len(debris_gdf)} polygons  ·  {debris_area_km2:.3f} km²",
    fontsize=10)
axes[1, 0].axis("off")

# [1,1] Post TCI + 三層疊加
axes[1, 1].imshow(rgb_post)
axes[1, 1].imshow(np.where(lake_mask.values,      1, np.nan), cmap="cool",   alpha=0.55)
axes[1, 1].imshow(np.where(landslide_mask.values, 1, np.nan), cmap="Reds",   alpha=0.55)
axes[1, 1].imshow(np.where(debris_mask.values,    1, np.nan), cmap="copper", alpha=0.55)

from matplotlib.patches import Patch
axes[1, 1].legend(handles=[
    Patch(facecolor="cyan",   alpha=0.7, label="Barrier lake"),
    Patch(facecolor="red",    alpha=0.7, label="Landslide source"),
    Patch(facecolor="sienna", alpha=0.7, label="Debris flow"),
], loc="lower right", fontsize=8, framealpha=0.8)
axes[1, 1].set_title("All three overlayed\n（空間互斥確認）", fontsize=10)
axes[1, 1].axis("off")

plt.tight_layout()
plt.savefig("output/three_masks.png", dpi=150, bbox_inches="tight")
plt.show()

# --- 空間互斥確認 ---
print(f"\n重疊像素數（應全為 0）:")
print(f"  lake ∩ landslide:  {int((lake_mask & landslide_mask).sum())}")
print(f"  lake ∩ debris:     {int((lake_mask & debris_mask).sum())}")
print(f"  landslide ∩ debris:{int((landslide_mask & debris_mask).sum())}")


### Cell 15 - 載入歷史資料層（W3 / W7 / W8）
---
本步驟整合前七週的分析成果，作為 ARIA v5.0 多層稽核的空間比對基準：

- **W3 避難所**：花蓮縣避難收容處所，含地形風險評估（risk_level）
- **W7 Top-5 瓶頸**：道路網路中介中心性最高的五個節點，代表災時最關鍵的交通要衝
- **W8 Guangfu Overlay**：光復鄉關鍵節點（車站、學校、鄉公所、橋梁、警戒區）

三個資料層統一重投影至 **EPSG:32651（UTM 51N）**，與衛星影像 cube 座標系一致，確保後續空間交叉分析的準確性。

In [ ]:
# [S11] 載入 W3 避難所、W7 瓶頸、W8 Guangfu overlay
import geopandas as gpd
import pandas as pd
from shapely import wkt

target_crs = "EPSG:32651"

# ── W3 避難所 ──────────────────────────────────────────────
shelters = pd.read_csv("data/gdf_shelter_buffer_500_risk_all.csv")
shelters["geometry"] = gpd.points_from_xy(shelters["經度"], shelters["緯度"])
shelters = gpd.GeoDataFrame(shelters, geometry="geometry", crs="EPSG:4326")
shelters = shelters.to_crs(target_crs)
print(f"W3 避難所: {len(shelters)} 筆")

# ── W7 Top-5 瓶頸 ──────────────────────────────────────────
top5 = pd.read_csv("data/top5_betweenness.csv")
top5["geometry"] = top5["geometry"].apply(wkt.loads)
top5 = gpd.GeoDataFrame(top5, geometry="geometry", crs=target_crs)
print(f"W7 Top-5 瓶頸: {len(top5)} 筆")

# ── W8 Guangfu overlay ─────────────────────────────────────
guangfu = gpd.read_file("data/guangfu_overlay.gpkg").to_crs(target_crs)
print(f"W8 Guangfu: {len(guangfu)} 筆")

print(f"\n全部 CRS = EPSG:{target_crs.split(':')[1]} ✅")

### Cell 16 - 多層空間稽核 — Eyewitness Impact Table
---
本步驟將三個偵測遮罩（堰塞湖、崩塌源區、土石流）與前週的避難所、道路瓶頸、光復節點進行空間交叉，產生正式的影響稽核表，量化 ARIA v4.0 的覆蓋缺口。

**命中規則：**
- 堰塞湖 / 土石流：點位落在偵測多邊形**內部**
- 崩塌源區：點位在偵測多邊形 **200m 緩衝**範圍內（考量 10m 像素邊界誤差）

In [ ]:
from shapely.ops import unary_union
from pyproj import Transformer

# ── 關鍵修正 1：debris_gdf 做事後 longitude gate ──────────────────────
tf_inv = Transformer.from_crs("EPSG:32651", "EPSG:4326", always_xy=True)

def centroid_lon(geom):
    lon, _ = tf_inv.transform(geom.centroid.x, geom.centroid.y)
    return lon

# 光復平原西緣 ~121.38°E，比 raster gate 121.35 更嚴格
debris_gdf_gated = debris_gdf[
    debris_gdf.geometry.apply(centroid_lon) > 121.38
].copy()

# ── 關鍵修正 2：預先建立 union（只算一次）────────────────────────────
lake_union        = unary_union(lake_gdf.geometry)             if len(lake_gdf) > 0           else None
landslide_union   = unary_union(landslides_gdf.geometry)       if len(landslides_gdf) > 0     else None
debris_union      = unary_union(debris_gdf_gated.geometry)     if len(debris_gdf_gated) > 0   else None

# 作業規定：shelter=100m, bottleneck=200m
ls_buf_100 = landslide_union.buffer(100) if landslide_union else None
ls_buf_200 = landslide_union.buffer(200) if landslide_union else None

# ── 關鍵修正 3：單一乾淨的 hit 函式 ──────────────────────────────────
def hit(point_geom, union):
    """'Y' if point touches/intersects union polygon, else 'N'"""
    if union is None or union.is_empty:
        return "N"
    return "Y" if point_geom.intersects(union) else "N"

# ── 建立 rows ────────────────────────────────────────────────────────
rows = []

# W3 避難所（landslide 100m 緩衝）
for _, r in shelters.iterrows():
    g = r.geometry
    rows.append({
        "Asset":              r.get("避難收容處所名稱", r.get("name", "shelter")),
        "Type":               "W3_shelter",
        "Location":           r.get("村里", "花蓮市"),
        "W4_terrain_risk":    r.get("risk_level", "—"),
        "W7_centrality_rank": "—",
        "lake_hit":           hit(g, lake_union),
        "landslide_hit":      hit(g, ls_buf_100),
        "debris_hit":         hit(g, debris_union),
        "Notes":              "",
    })

# W7 瓶頸（landslide 200m 緩衝）
for _, r in top5.iterrows():
    g = r.geometry
    rows.append({
        "Asset":              r.get("name", f"Node_{r.get('node_id','')}"),
        "Type":               "W7_bottleneck",
        "Location":           "台9線",
        "W4_terrain_risk":    r.get("terrain_risk", "—"),
        "W7_centrality_rank": r.get("rank", "—"),
        "lake_hit":           hit(g, lake_union),
        "landslide_hit":      hit(g, ls_buf_200),
        "debris_hit":         hit(g, debris_union),
        "Notes":              "",
    })

# W8 Guangfu overlay（landslide 100m 緩衝）
for _, r in guangfu.iterrows():
    g = r.geometry
    rows.append({
        "Asset":              r.get("cn_name", r.get("name", "node")),
        "Type":               f"W8_{r.get('node_type','node')}",
        "Location":           "光復鄉",
        "W4_terrain_risk":    "—",
        "W7_centrality_rank": "—",
        "lake_hit":           hit(g, lake_union),
        "landslide_hit":      hit(g, ls_buf_100),
        "debris_hit":         hit(g, debris_union),
        "Notes":              r.get("node_type", ""),
    })

# ── 排序 + Notes ────────────────────────────────────────────────────
impact_df = pd.DataFrame(rows)

_risk_ord = {"extremely_high": 0, "high": 1, "medium": 2, "low": 3}
impact_df["_r"] = impact_df["W4_terrain_risk"].map(lambda x: _risk_ord.get(x, 9))
impact_df = (
    impact_df
    .sort_values(["debris_hit", "landslide_hit", "_r"],
                 ascending=[False, False, True])
    .drop(columns="_r")
    .reset_index(drop=True)
)

def assign_notes(row):
    if row["landslide_hit"] == "Y" and row["debris_hit"] == "Y":
        return "collapsed Sep 23 16:00"
    elif row["debris_hit"] == "Y":
        return "buried in 20 cm debris"
    elif row["landslide_hit"] == "Y":
        return "landslide affected"
    elif row["lake_hit"] == "Y":
        return "within barrier lake zone"
    else:
        return "outside event area"

impact_df["Notes"] = impact_df.apply(assign_notes, axis=1)
impact_df.to_csv("impact_table.csv", index=False)
print(impact_df.to_markdown(index=False))
print(f"\nGuangfu hits: {(impact_df[impact_df['Type'].str.startswith('W8')]['debris_hit']=='Y').sum()} / 5")

### Cell 17 - 終極影響圖 — ARIA v5.0 三幕稽核總圖
---
整合三幕災害偵測結果（堰塞湖、崩塌源區、土石流）與基礎設施（W3避難所、W7瓶頸、W8光復節點），產生馬太鞍溪堰塞湖事件的完整影響評估圖。以事件後Sentinel-2真彩影像為底圖，疊加空間互斥的三類災害遮罩，提供災後評估與決策支援的視覺化依據。


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

rgb_post = composite_stretched(cube_post, "B04", "B03", "B02")
left, bottom, right, top_bound = cube_post.rio.bounds()

fig, ax = plt.subplots(figsize=(10, 10))

# ── 底圖 Post TCI ──
ax.imshow(
    rgb_post,
    extent=[left, right, bottom, top_bound],
    origin="upper",
    alpha=0.85,
    zorder=1
)

# ── 三個偵測遮罩：改淡一點，邊界清楚一點 ──
lake_gdf.plot(
    ax=ax,
    facecolor="deepskyblue",
    edgecolor="deepskyblue",
    linewidth=0.8,
    alpha=0.45,
    zorder=2
)

landslides_gdf.plot(
    ax=ax,
    facecolor="red",
    edgecolor="darkred",
    linewidth=0.8,
    alpha=0.30,
    zorder=3
)

debris_gdf.plot(
    ax=ax,
    facecolor="peru",
    edgecolor="saddlebrown",
    linewidth=0.6,
    alpha=0.22,
    zorder=2
)

# ── W3 避難所：縮小 ──
shelters.plot(
    ax=ax,
    color="limegreen",
    marker="o",
    markersize=12,   # 原本 30，先縮到 12
    edgecolor="black",
    linewidth=0.3,
    zorder=5
)

# ── W7 Top-5 瓶頸 ──
top5.plot(
    ax=ax,
    color="gold",
    marker="D",
    markersize=42,
    edgecolor="black",
    linewidth=0.5,
    zorder=6
)

# ── W8 Guangfu 節點 ──
guangfu.plot(
    ax=ax,
    color="darkred",
    marker="*",
    markersize=120,
    edgecolor="black",
    linewidth=0.5,
    zorder=7
)

# ── Guangfu 節點標籤：字再小一點，框更淡 ──
for _, r in guangfu.iterrows():
    ax.annotate(
        r.get("name", ""),
        xy=(r.geometry.x, r.geometry.y),
        xytext=(5, 3),
        textcoords="offset points",
        fontsize=7,
        color="darkred",
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.18",
            fc="white",
            ec="none",
            alpha=0.75
        ),
        zorder=8
    )

# ── 命中統計文字框 ──
n_debris = int((impact_df["debris_hit"] == "Y").sum())
n_land   = int((impact_df["landslide_hit"] == "Y").sum())
n_lake   = int((impact_df["lake_hit"] == "Y").sum())

ax.text(
    0.02, 0.03,
    f"Lake hits: {n_lake} | Landslide hits: {n_land} | Debris hits: {n_debris}",
    transform=ax.transAxes,
    fontsize=10,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="white",
        edgecolor="0.3",
        alpha=0.9
    ),
    zorder=10
)

# ── 自訂 legend：比直接用 plot label 好看 ──
legend_handles = [
    mpatches.Patch(facecolor="deepskyblue", edgecolor="deepskyblue", alpha=0.45, label="Barrier Lake"),
    mpatches.Patch(facecolor="red", edgecolor="darkred", alpha=0.30, label="Landslide Source"),
    mpatches.Patch(facecolor="peru", edgecolor="saddlebrown", alpha=0.22, label="Debris Flow"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="limegreen",
           markeredgecolor="black", markeredgewidth=0.4, markersize=6, label="W3 Shelters"),
    Line2D([0], [0], marker="D", color="w", markerfacecolor="gold",
           markeredgecolor="black", markeredgewidth=0.5, markersize=7, label="W7 Top-5 Bottlenecks"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="darkred",
           markeredgecolor="black", markeredgewidth=0.5, markersize=12, label="W8 Guangfu Overlay"),
]

ax.legend(
    handles=legend_handles,
    loc="upper right",
    fontsize=8.5,
    framealpha=0.92,
    facecolor="white",
    edgecolor="0.7"
)

# ── 版面設定 ──
ax.set_title(
    "ARIA v5.0 — Final Impact Map\n馬太鞍溪堰塞湖事件 三幕稽核總圖",
    fontsize=15,
    fontweight="bold",
    pad=12
)
ax.set_xlabel("UTM Easting (m)", fontsize=11)
ax.set_ylabel("UTM Northing (m)", fontsize=11)

# 可選：不想太擠可以固定比例
ax.set_aspect("equal")

# 讓邊框乾淨一點
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("0.3")

plt.tight_layout()
plt.savefig("output/final_impact_map.png", dpi=180, bbox_inches="tight")
plt.show()

print("✅ 已儲存：output/final_impact_map.png")

# 🛰️ 花蓮縣災害防救指揮中心  
## 機密應變簡報 — 呈縣長  

**簡報人：作戰科長**  
**日期：2025年10月（Post 場景日）**

---

## 一、三幕衛星影像確認時序

ARIA v5.0 三景 Sentinel-2 L2A 影像提供完整事件鏈證據：

- **ACT 1（2025年6月）**  
  馬太鞍溪上游植被完整、無異常水體，作為事件前基準。  

- **ACT 2（2025年8月）**  
  座標 (121.292°E, 23.696°N) 出現 **0.854 km² 混濁堰塞湖**，  
  NIR 反射率 0.10–0.18，顯示高懸沙濃度水體 → 土壩已形成。  

- **ACT 3（2025年10月）**  
  湖體消失，上游崩塌源區 **4.039 km²**，  
  下游光復平原土石流覆蓋 **{debris_area_km2:.3f} km²**，  
  橋梁與農地受損確認。  

---

## 二、預警窗口（64天）

**期間：2025-07-21 ～ 2025-09-23**

- Sentinel-2 重訪：約 5 天  
- 預估可用影像：8 月間至少 **3–4 景（雲量 < 40%）**  

**關鍵節點：**
- 8 月中旬：湖面 > 0.3 km² → 可觸發首次預警  
- 9 月 11 日：湖面達峰值 **0.86 km²** → 應發布最高級警報  

➡️ 可爭取約 **40 天工程應對與疏散時間**

---

## 三、覆蓋缺口

- ARIA W3/W7 系統以花蓮市為核心  
- 與光復鄉距離：約 **40 km（未覆蓋）**  

**影響評估：**
- 光復鄉 5 個關鍵節點：**{n_guangfu_hits}/5 受影響**
- W3 避難所：**0 命中**
- W7 瓶頸點：**0 命中**

**結論：**
需擴展監測至：
- 萬榮鄉
- 光復鄉
- 瑞穗鄉  

並建立：  
👉「中央山脈東麓堰塞湖監測圈」

---

## 四、未來 24 小時作戰令

| 優先序 | 任務 | 目標 | 單位 |
|--------|------|------|------|
| 1 | 清除 | 台9線馬太鞍溪橋土石，恢復交通 | 工兵第三群 |
| 2 | 補給 | 光復國小避難所 72 小時物資 | 後勤支援隊 |
| 3 | UAV 偵察 | 佛祖街沉積區前緣監測 | 空中偵搜分隊 |
| 4 | 人口清查 | 大富、大豐部落失聯確認 | 民政局 |

---

## 五、模型精進建議（ARIA v6.0）

導入「堰塞湖自動監測機制」：

- 每次新影像 → 自動比對前次結果  
- 觸發條件：
  - 面積增加 > 10%  
  - 或總面積 > 0.3 km²  

➡️ 即時推播至災防平台  

**預期效益：**
- 可提前 **30–40 天** 預警潰決風險  
- 顯著提升防災決策時效  

---

📌 **總結**  
本事件顯示，透過衛星時序分析可明確掌握「崩塌 → 堰塞湖 → 潰決 → 土石流」完整鏈條。未來應強化監測覆蓋與自動預警機制，以避免類似災害造成更大損失。

### Coverage Gap 分析 — ARIA 系統覆蓋範圍的關鍵缺失

#### 觀察結果

本次三幕稽核結果顯示了一個重要的 **coverage gap（覆蓋缺口）**：

**土石流影響範圍**：
- ACT 1→3 偵測到的土石流面積約 **83.392 km²**
- 主要覆蓋光復鄉下游農田與聚落區域
- 影響範圍集中於馬太鞍溪出海口以東（121.38°E 以東）

**ARIA 前期系統覆蓋狀況**：
- **Week 3 避難所**：主要位於花蓮市區，距離光復鄉約 20-30 公里
- **Week 7 道路瓶頸**：集中於蘇花公路與花蓮市交通要道
- **命中統計**：W3/W7 設施幾乎 **零命中**（0% 被土石流影響）

#### 覆蓋缺口的根本原因

1. **地理範圍限制**：
   - ARIA v3.0-v4.0 主要以花蓮市為核心
   - 未涵蓋光復鄉、豐濱鄉等東部沿海鄉鎮

2. **災害鏈認知不足**：
   - 前期系統專注於直接災害（地震、降雨）
   - 未考慮上游崩塌→堰塞湖→下游土石流的連鎖效應

3. **網路分析範圍**：
   - W7 道路網路分析僅限主要省道
   - 未包含光復鄉內部次要道路與鄉村道路

#### 教訓與建議

**立即行動**：
- 擴展 ARIA 系統至整個花蓮縣，特別是萬榮、光復、豐濱等高風險鄉鎮
- 建立上游-下游災害鏈結模型，納入堰塞湖潰決風險評估

**長期改進**：
- 整合高解析度地形資料，提升上游崩塌潛勢分析
- 建立鄉村級道路網路，擴大交通脆弱性評估範圍
- 開發跨鄉鎮疏散路線規劃，考慮災害鏈結效應

#### 結論

本次馬太鞍溪事件清楚揭示了 ARIA 系統的 **地理覆蓋缺口**。雖然系統在花蓮市區運作良好，但對於上游山區引發的下游災害缺乏預警能力。這提醒我們：**災害風險評估必須考慮完整的災害鏈結，而非僅聚焦於人口密集區**。

未來 ARIA 系統升级應優先考慮：
1. **全縣覆蓋**：擴展至所有鄉鎮
2. **災害鏈建模**：整合上游至下游的連鎖效應  
3. **鄉村基礎設施**：納入更多鄉村級關鍵設施

只有這樣，ARIA 系統才能真正成為花蓮縣全方位的災害預防與應決策支援工具。